# 多层感知机的从零开始实现
:label:`sec_mlp_scratch`

我们已经在 :numref:`sec_mlp`中描述了多层感知机（MLP），
现在让我们尝试自己实现一个多层感知机。
为了与之前softmax回归（ :numref:`sec_softmax_scratch` ）
获得的结果进行比较，
我们将继续使用Fashion-MNIST图像分类数据集
（ :numref:`sec_fashion_mnist`）。


In [ ]:
%pip install -q d2l==0.17.6 --no-deps


In [19]:
import torch
from torch import nn
from d2l import torch as d2l

In [20]:
batch_size = 256
train_iter, test_iter = d2l.load_data_fashion_mnist(batch_size)

## 初始化模型参数

回想一下，Fashion-MNIST中的每个图像由
$28 \times 28 = 784$个灰度像素值组成。
所有图像共分为10个类别。
忽略像素之间的空间结构，
我们可以将每个图像视为具有784个输入特征
和10个类的简单分类数据集。
首先，我们将[**实现一个具有单隐藏层的多层感知机，
它包含256个隐藏单元**]。
注意，我们可以将这两个变量都视为超参数。
通常，我们选择2的若干次幂作为层的宽度。
因为内存在硬件中的分配和寻址方式，这么做往往可以在计算上更高效。

我们用几个张量来表示我们的参数。
注意，对于每一层我们都要记录一个权重矩阵和一个偏置向量。
跟以前一样，我们要为损失关于这些参数的梯度分配内存。


In [83]:
num_inputs, num_outputs, num_hiddens = 784, 10, 512看起来其实你的

W1 = nn.Parameter(torch.randn(
    num_inputs, num_hiddens, requires_grad=True) * 0.01)
b1 = nn.Parameter(torch.zeros(num_hiddens, requires_grad=True))
W2 = nn.Parameter(torch.randn(
    num_hiddens, num_outputs, requires_grad=True) * 0.01)
b2 = nn.Parameter(torch.zeros(num_outputs, requires_grad=True))

params = [W1, b1, W2, b2]

## 激活函数

为了确保我们对模型的细节了如指掌，
我们将[**实现ReLU激活函数**]，
而不是直接调用内置的`relu`函数。


In [84]:
def relu(X):
    a = torch.zeros_like(X)
    return torch.max(X, a)

## 模型

因为我们忽略了空间结构，
所以我们使用`reshape`将每个二维图像转换为一个长度为`num_inputs`的向量。
只需几行代码就可以(**实现我们的模型**)。


In [85]:
def net(X):
    X = X.reshape((-1, num_inputs))
    H = relu(X@W1 + b1)  # 这里“@”代表矩阵乘法
    return (H@W2 + b2)

## 损失函数

由于我们已经从零实现过softmax函数（ :numref:`sec_softmax_scratch`），
因此在这里我们直接使用高级API中的内置函数来计算softmax和交叉熵损失。
回想一下我们之前在 :numref:`subsec_softmax-implementation-revisited`中
对这些复杂问题的讨论。
我们鼓励感兴趣的读者查看损失函数的源代码，以加深对实现细节的了解。


In [86]:
loss = nn.CrossEntropyLoss(reduction='none')

## 训练

幸运的是，[**多层感知机的训练过程与softmax回归的训练过程完全相同**]。
可以直接调用`d2l`包的`train_ch3`函数（参见 :numref:`sec_softmax_scratch` ），
将迭代周期数设置为10，并将学习率设置为0.1.


In [88]:
num_epochs, lr = 50, 0.1
updater = torch.optim.SGD(params, lr=lr)

for epoch in range(num_epochs):
    train_loss, train_acc = d2l.train_epoch_ch3(
        net, train_iter, loss, updater)
    test_acc = d2l.evaluate_accuracy(net, test_iter)
    print(f'epoch {epoch + 1:02d}: '
          f'train loss {train_loss:.4f}, '
          f'train acc {train_acc:.4f}, '
          f'test acc {test_acc:.4f}')

epoch 01: train loss 0.2271, train acc 0.9198, test acc 0.8793
epoch 02: train loss 0.2239, train acc 0.9205, test acc 0.8870
epoch 03: train loss 0.2256, train acc 0.9184, test acc 0.8743
epoch 04: train loss 0.2207, train acc 0.9213, test acc 0.8232
epoch 05: train loss 0.2191, train acc 0.9230, test acc 0.8724
epoch 06: train loss 0.2162, train acc 0.9228, test acc 0.8775
epoch 07: train loss 0.2147, train acc 0.9234, test acc 0.8711
epoch 08: train loss 0.2138, train acc 0.9239, test acc 0.8846
epoch 09: train loss 0.2111, train acc 0.9249, test acc 0.8652
epoch 10: train loss 0.2095, train acc 0.9254, test acc 0.8877
epoch 11: train loss 0.2073, train acc 0.9263, test acc 0.8874
epoch 12: train loss 0.2058, train acc 0.9271, test acc 0.8837
epoch 13: train loss 0.2030, train acc 0.9281, test acc 0.8853
epoch 14: train loss 0.2007, train acc 0.9282, test acc 0.8783
epoch 15: train loss 0.2000, train acc 0.9293, test acc 0.8719
epoch 16: train loss 0.1993, train acc 0.9292, test acc

为了对学习到的模型进行评估，我们将[**在一些测试数据上应用这个模型**]。


[Discussions](https://discuss.d2l.ai/t/1804)
